# 🔬 DermaLens: Dual-Backbone Hybrid Skin Lesion Classifier
### End-to-End Training, Evaluation & Visual Explainability in Google Colab

> [!CAUTION]
> **UNDER ACTIVE DEVELOPMENT — RESEARCH PROTOTYPE**
> DermaLens is an investigational screening-support prototype. It does not constitute a medical diagnosis.

This notebook packages the complete DermaLens training pipeline into a single, self-contained environment:
1. **Dual Visual Backbones:** EfficientNet-B4 (17.5M) + Swin Transformer V2 (86.9M)
2. **Clinical Patient Metadata MLP:** 13-dimensional one-hot encoded metadata (Age, Sex, Anatomical Site)
3. **Loss & Regularization:** Multi-class Focal Loss, Label Smoothing, CutMix augmentation, and Auxiliary Metadata Loss
4. **Optimization:** AdamW with Swin layer-wise LR decay, Cosine Annealing, and Stochastic Weight Averaging (SWA)
5. **Evaluation & Explainability:** Full classification report, confusion matrix, and Grad-CAM++ attention heatmaps

---
**Recommended Runtime:** `Runtime` → `Change runtime type` → **T4 GPU** (or A100/V100).


In [ ]:
# Step 1: Install required dependencies
!pip install -q timm albumentations opencv-python-headless scikit-learn pandas matplotlib seaborn

import os
import sys
import time
import random
import json
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.optim.swa_utils import AveragedModel, SWALR

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score

# Check CUDA device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ Compute Device : {device}")
if torch.cuda.is_available():
    print(f"✓ GPU Model      : {torch.cuda.get_device_name(0)}")
    print(f"✓ Device Memory  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ Warning: Running on CPU. For fast training, enable GPU under Runtime -> Change runtime type.")


In [ ]:
# Step 2: Global Configuration & Hyperparameters
class Config:
    # Diagnostic Classes (ISIC 2019 8-class standard)
    CLASS_NAMES = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]
    NUM_CLASSES = len(CLASS_NAMES)
    CLASS_FULL = {
        "MEL": "Melanoma",
        "NV": "Melanocytic Nevus",
        "BCC": "Basal Cell Carcinoma",
        "AK": "Actinic Keratosis",
        "BKL": "Benign Keratosis",
        "DF": "Dermatofibroma",
        "VASC": "Vascular Lesion",
        "SCC": "Squamous Cell Carcinoma",
    }
    ALL_SITES = [
        "anterior torso", "head/neck", "lateral torso", "lower extremity",
        "oral/genital", "palms/soles", "posterior torso", "upper extremity", "unknown"
    ]
    
    # Image & Batch Settings
    IMG_SIZE = 384
    BATCH_SIZE = 8             # Optimized for Colab 15GB GPU
    NUM_WORKERS = 2
    GRAD_ACCUM_STEPS = 2      # Effective batch size = 16
    
    # Training Duration & Learning Rates
    EPOCHS = 10                # Set to 50 for full production training
    LR = 1e-4
    WEIGHT_DECAY = 0.01
    SWIN_LR_DECAY = 0.65       # Layer-wise decay for Swin Transformer
    COSINE_T_MAX = 10
    SWA_START = 6              # Stochastic Weight Averaging start epoch
    
    # Loss & Regularization
    FOCAL_GAMMA = 2.0
    FOCAL_ALPHA = [1.0] * 8
    LABEL_SMOOTHING = 0.1
    CUTMIX_PROB = 0.30
    META_AUX_WEIGHT = 0.3      # Weight of auxiliary loss on metadata branch
    
    # Preprocessing
    AGE_NORM_FACTOR = 90.0     # Standardized min-max normalization factor
    USE_AMP = torch.cuda.is_available()
    SEED = 42
    OUTPUT_DIR = Path("./dermalens_output")

Config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed=Config.SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed()
print("✓ Hyperparameters and seed initialized.")


### Step 3: Dataset Setup Options
You can load your dataset using any of the three options below:
* **Option A: Google Drive** — If you have ISIC 2019 uploaded to your Drive.
* **Option B: Kaggle API** — Download directly from Kaggle into the Colab VM.
* **Option C: Quick-Start Synthetic Generator** — Automatically builds a mini dataset with synthetic dermoscopy patterns so you can test and run this entire notebook immediately!


In [ ]:
# Option A: Mount Google Drive (Uncomment if using Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = "/content/drive/MyDrive/DermaLens_Data"

# Option B: Kaggle Download (Uncomment if using Kaggle)
# !pip install -q kaggle
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle competitions download -c isic-2019 -p ./isic2019 --unzip

# Option C: Automated Quick-Start Dataset Generator
# Generates realistic synthetic lesion images with clinical metadata manifests for immediate execution.
DATA_ROOT = Path("./dataset_demo")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

def generate_synthetic_lesion(class_name: str, size=(Config.IMG_SIZE, Config.IMG_SIZE)) -> np.ndarray:
    """Synthesizes a realistic dermoscopy lesion pattern with pigment network and peripheral skin."""
    h, w = size
    # Base skin background (warm flesh tone with slight texture)
    base_color = np.array([random.randint(180, 230), random.randint(140, 185), random.randint(120, 160)], dtype=np.uint8)
    img = np.full((h, w, 3), base_color, dtype=np.uint8)
    
    # Noise/skin texture
    noise = np.random.normal(0, 10, (h, w, 3)).astype(np.int16)
    img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    
    # Elliptical lesion center
    center = (w // 2 + random.randint(-20, 20), h // 2 + random.randint(-20, 20))
    axes = (random.randint(60, 110), random.randint(50, 95))
    angle = random.randint(0, 180)
    
    # Distinct diagnostic shades
    if class_name in ("MEL", "NV"):
        lesion_color = (random.randint(30, 70), random.randint(20, 50), random.randint(10, 40)) # dark pigment
    elif class_name in ("BCC", "SCC"):
        lesion_color = (random.randint(120, 160), random.randint(60, 90), random.randint(70, 100)) # reddish ulcer/plaque
    elif class_name == "VASC":
        lesion_color = (random.randint(140, 190), random.randint(20, 50), random.randint(40, 70)) # purple/red vascular
    else:
        lesion_color = (random.randint(80, 120), random.randint(60, 90), random.randint(40, 60)) # keratotic brown
        
    cv2.ellipse(img, center, axes, angle, 0, 360, lesion_color, -1)
    img = cv2.GaussianBlur(img, (25, 25), 0)
    return img

print("Building test dataset splits (Train/Val/Test)...")
records = []
for split, count in [("train", 120), ("val", 32), ("test", 32)]:
    split_dir = DATA_ROOT / split
    split_dir.mkdir(parents=True, exist_ok=True)
    for i in range(count):
        cls_idx = i % Config.NUM_CLASSES
        cls_name = Config.CLASS_NAMES[cls_idx]
        img_arr = generate_synthetic_lesion(cls_name)
        img_name = f"ISIC_{split}_{i:04d}_{cls_name}.jpg"
        img_path = split_dir / img_name
        cv2.imwrite(str(img_path), cv2.cvtColor(img_arr, cv2.COLOR_RGB2BGR))
        
        # Clinical metadata
        age = random.choice([25, 40, 55, 65, 75, np.nan])
        sex = random.choice(["male", "female", "unknown"])
        site = random.choice(Config.ALL_SITES)
        
        records.append({
            "split": split,
            "image_path": str(img_path.resolve()),
            "class_encoded": cls_idx,
            "target": cls_name,
            "age_approx": age,
            "sex": sex,
            "site": site,
        })

df_all = pd.DataFrame(records)
df_train = df_all[df_all["split"] == "train"].drop(columns=["split"])
df_val   = df_all[df_all["split"] == "val"].drop(columns=["split"])
df_test  = df_all[df_all["split"] == "test"].drop(columns=["split"])

train_csv = DATA_ROOT / "train.csv"
val_csv   = DATA_ROOT / "val.csv"
test_csv  = DATA_ROOT / "test.csv"

df_train.to_csv(train_csv, index=False)
df_val.to_csv(val_csv, index=False)
df_test.to_csv(test_csv, index=False)

print(f"✓ Synthetic test dataset created at: {DATA_ROOT}")
print(f"  Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")


In [ ]:
# Step 4: Data Augmentation & Dataloaders
TRAIN_TRANSFORMS = A.Compose([
    A.Resize(Config.IMG_SIZE, Config.IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=45, p=0.6),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
    A.GaussianBlur(blur_limit=(3, 7), p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

VAL_TRANSFORMS = A.Compose([
    A.Resize(Config.IMG_SIZE, Config.IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

def encode_metadata(age=None, sex=None, site=None) -> torch.Tensor:
    """Encodes tabular metadata into a normalized 13-dimensional tensor."""
    meta = np.zeros(13, dtype=np.float32)
    meta[0] = min(max(float(age), 0.0), Config.AGE_NORM_FACTOR) / Config.AGE_NORM_FACTOR if age is not None and not np.isnan(age) else 0.5
    sex_str = str(sex or "unknown").lower()
    if sex_str == "female":
        meta[1] = 1.0
    elif sex_str == "male":
        meta[2] = 1.0
    else:
        meta[3] = 1.0
        
    site_str = str(site or "unknown").lower()
    matched = False
    for i, s in enumerate(Config.ALL_SITES):
        if s == site_str:
            meta[4 + i] = 1.0
            matched = True
            break
    if not matched:
        meta[4 + Config.ALL_SITES.index("unknown")] = 1.0
    return torch.tensor(meta, dtype=torch.float32)

class DermaLensDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(row["image_path"])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if self.transform:
            image = self.transform(image=image)["image"]
            
        meta = encode_metadata(row.get("age_approx"), row.get("sex"), row.get("site"))
        label = torch.tensor(int(row["class_encoded"])).long()
        return image, meta, label

# Dataloaders with WeightedRandomSampler for class balance
train_dataset = DermaLensDataset(train_csv, transform=TRAIN_TRANSFORMS)
val_dataset   = DermaLensDataset(val_csv, transform=VAL_TRANSFORMS)
test_dataset  = DermaLensDataset(test_csv, transform=VAL_TRANSFORMS)

labels = train_dataset.df["class_encoded"].values
class_counts = np.bincount(labels, minlength=Config.NUM_CLASSES)
class_weights = 1.0 / np.maximum(class_counts, 1)
sample_weights = class_weights[labels]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, sampler=sampler, num_workers=Config.NUM_WORKERS)
val_loader   = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS)
test_loader  = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS)

print(f"✓ Dataloaders ready: {len(train_loader)} batches train | {len(val_loader)} batches val")


In [ ]:
# Step 5: DermaLens Hybrid Model Architecture
class DermaLensHybrid(nn.Module):
    def __init__(self, num_classes=Config.NUM_CLASSES, meta_dim=13, mode='full', pretrained=True):
        super().__init__()
        self.mode = mode
        
        # 1. EfficientNet-B4 Convolutional Backbone (1792-d)
        if mode in ('effnet_only', 'image_only', 'full'):
            self.effnet = timm.create_model('efficientnet_b4', pretrained=pretrained, num_classes=0)
            
        # 2. Swin Transformer V2 Backbone (1024-d)
        if mode in ('swin_only', 'image_only', 'full'):
            self.swin = timm.create_model('swinv2_base_window12to24_192to384.ms_in22k_ft_in1k', pretrained=pretrained, num_classes=0)
            
        # 3. Clinical Metadata MLP (13 -> 64 -> 32-d)
        if mode == 'full':
            self.meta_mlp = nn.Sequential(
                nn.Linear(meta_dim, 64),
                nn.BatchNorm1d(64),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(64, 32),
                nn.ReLU()
            )
            self.meta_aux_head = nn.Linear(32, num_classes) # Auxiliary loss supervisor
            
        fusion_dim = {
            'effnet_only': 1792,
            'swin_only': 1024,
            'image_only': 1792 + 1024,
            'full': 1792 + 1024 + 32
        }[mode]
        
        # 4. Late-Fusion Classification Head
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
        
        if mode in ('image_only', 'full'):
            self.freeze_backbones()
            
    def freeze_backbones(self):
        """Freeze early backbone layers to avoid catastrophic forgetting during transfer learning."""
        if hasattr(self, 'effnet'):
            for name, param in self.effnet.named_parameters():
                if not name.startswith('blocks.6') and not name.startswith('conv_head') and not name.startswith('bn2'):
                    param.requires_grad = False
        if hasattr(self, 'swin'):
            for name, param in self.swin.named_parameters():
                if not name.startswith('layers.3') and not name.startswith('norm'):
                    param.requires_grad = False
                    
    def forward(self, img, meta=None, return_aux=False):
        features = []
        aux_logits = None
        
        if self.mode in ('effnet_only', 'image_only', 'full'):
            features.append(self.effnet(img))
        if self.mode in ('swin_only', 'image_only', 'full'):
            features.append(self.swin(img))
        if self.mode == 'full':
            if meta is None:
                meta = torch.zeros(img.size(0), 13, device=img.device)
            meta_feat = self.meta_mlp(meta)
            features.append(meta_feat)
            if hasattr(self, 'meta_aux_head'):
                aux_logits = self.meta_aux_head(meta_feat)
                
        logits = self.classifier(torch.cat(features, dim=1))
        if return_aux:
            return logits, aux_logits
        return logits

# Instantiate Model
model = DermaLensHybrid(num_classes=Config.NUM_CLASSES, mode='full', pretrained=True).to(device)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"✓ Model initialized | Trainable parameters: {trainable_params/1e6:.2f}M / {total_params/1e6:.2f}M total")


In [ ]:
# Step 6: Loss Functions, CutMix & Optimizer
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.1):
        super().__init__()
        self.gamma = gamma
        self.alpha = torch.tensor(alpha or [1.0]*8, dtype=torch.float32).to(device)
        self.smoothing = label_smoothing

    def forward(self, logits, targets):
        num_classes = logits.size(-1)
        with torch.no_grad():
            smooth_targets = torch.full_like(logits, self.smoothing / (num_classes - 1))
            smooth_targets.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        log_p = F.log_softmax(logits, dim=1)
        p = torch.exp(log_p)
        focal_weight = (1.0 - p) ** self.gamma
        if self.alpha is not None:
            focal_weight = focal_weight * self.alpha.unsqueeze(0)
        return -(focal_weight * smooth_targets * log_p).sum(dim=1).mean()

def cutmix_data(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    bbx1, bby1, bbx2, bby2 = rand_bbox(x.size(), lam)
    x[:, :, bbx1:bbx2, bby1:bby2] = x[index, :, bbx1:bbx2, bby1:bby2]
    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (x.size()[-1] * x.size()[-2]))
    return x, y[index], lam

def rand_bbox(size, lam):
    W, H = size[2], size[3]
    cut_rat = np.sqrt(1.0 - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)
    return bbx1, bby1, bbx2, bby2

# Layer-wise Optimizer Setup
params = []
if hasattr(model, 'effnet'):
    params.append({'params': model.effnet.parameters(), 'lr': Config.LR * 0.1})
if hasattr(model, 'swin'):
    params.append({'params': model.swin.parameters(), 'lr': Config.LR * 0.1 * Config.SWIN_LR_DECAY})
if hasattr(model, 'meta_mlp'):
    params.append({'params': model.meta_mlp.parameters(), 'lr': Config.LR * 3.0})
if hasattr(model, 'meta_aux_head'):
    params.append({'params': model.meta_aux_head.parameters(), 'lr': Config.LR * 3.0})
params.append({'params': model.classifier.parameters(), 'lr': Config.LR})

optimizer = torch.optim.AdamW(params, weight_decay=Config.WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=Config.COSINE_T_MAX, eta_min=1e-6)
criterion = FocalLoss(gamma=Config.FOCAL_GAMMA, alpha=Config.FOCAL_ALPHA, label_smoothing=Config.LABEL_SMOOTHING)

# Stochastic Weight Averaging (SWA)
swa_model = AveragedModel(model)
swa_scheduler = SWALR(optimizer, swa_lr=Config.LR * 0.5)

print("✓ Optimizer, Schedulers, and Loss Criterion configured.")


In [ ]:
# Step 7: Complete Training Loop
def validate(net, loader):
    net.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, metas, targets in loader:
            imgs, metas = imgs.to(device), metas.to(device)
            outputs = net(imgs, metas)
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(targets.numpy())
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    return macro_f1, acc

print("🚀 Starting DermaLens Hybrid Model Training...")
scaler = torch.amp.GradScaler('cuda', enabled=Config.USE_AMP)
best_f1 = 0.0
best_model_path = Config.OUTPUT_DIR / "best_dermalens_model.pt"

for epoch in range(1, Config.EPOCHS + 1):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    
    for step, (images, metas, targets) in enumerate(train_loader):
        images, metas, targets = images.to(device), metas.to(device), targets.to(device)
        
        # Apply CutMix
        use_cutmix = random.random() < Config.CUTMIX_PROB
        if use_cutmix:
            images, targets_cut, lam = cutmix_data(images, targets)
        else:
            targets_cut, lam = targets, 1.0
            
        with torch.amp.autocast('cuda', enabled=Config.USE_AMP):
            output = model(images, metas, return_aux=True)
            logits, meta_aux = output if isinstance(output, tuple) else (output, None)
            
            # Loss calculation
            if use_cutmix:
                main_loss = lam * criterion(logits, targets) + (1 - lam) * criterion(logits, targets_cut)
            else:
                main_loss = criterion(logits, targets)
                
            loss = main_loss
            if meta_aux is not None:
                aux_loss = criterion(meta_aux, targets)
                loss = loss + Config.META_AUX_WEIGHT * aux_loss
                
            loss = loss / Config.GRAD_ACCUM_STEPS
            
        scaler.scale(loss).backward()
        total_loss += loss.item() * Config.GRAD_ACCUM_STEPS
        
        if (step + 1) % Config.GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            
    # SWA update
    if epoch >= Config.SWA_START:
        swa_model.update_parameters(model)
        swa_scheduler.step()
    else:
        scheduler.step()
        
    val_f1, val_acc = validate(model, val_loader)
    print(f"Epoch [{epoch:02d}/{Config.EPOCHS:02d}] - Loss: {total_loss/len(train_loader):.4f} | Val F1: {val_f1:.4f} | Val Acc: {val_acc*100:.2f}%")
    
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), best_model_path)
        print(f"  ⭐ Saved new best model checkpoint (Macro-F1: {best_f1:.4f})")

print(f"
🎉 Training complete! Best validation Macro-F1: {best_f1:.4f}")


In [ ]:
# Step 8: Comprehensive Test Set Evaluation
# Load best weights
model.load_state_dict(torch.load(best_model_path))
model.eval()

y_preds, y_trues, y_probs = [], [], []
with torch.no_grad():
    for imgs, metas, targets in test_loader:
        imgs, metas = imgs.to(device), metas.to(device)
        logits = model(imgs, metas)
        probs = F.softmax(logits, dim=1).cpu().numpy()
        preds = logits.argmax(dim=1).cpu().numpy()
        y_preds.extend(preds)
        y_trues.extend(targets.numpy())
        y_probs.extend(probs)

print("=" * 60)
print("           DERMALENS CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_trues, y_preds, target_names=Config.CLASS_NAMES, zero_division=0))

# Confusion Matrix Heatmap
cm = confusion_matrix(y_trues, y_preds)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=Config.CLASS_NAMES, yticklabels=Config.CLASS_NAMES)
plt.title("DermaLens Confusion Matrix (Test Set)")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.tight_layout()
plt.show()


In [ ]:
# Step 9: Visual Explainability (Grad-CAM++ on Test Image)
class GradCAMPlusPlus:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        target_layer.register_forward_hook(self.save_activation)
        target_layer.register_full_backward_hook(self.save_gradient)
        
    def save_activation(self, module, input, output):
        self.activations = output
        
    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]
        
    def generate(self, img_tensor, meta_tensor, target_class=None):
        self.model.eval()
        output = self.model(img_tensor, meta_tensor)
        if target_class is None:
            target_class = output.argmax(dim=1).item()
            
        self.model.zero_grad()
        score = output[0, target_class]
        score.backward(retain_graph=True)
        
        grads = self.gradients[0]
        acts = self.activations[0]
        
        # Grad-CAM++ weighting
        g2 = grads ** 2
        g3 = grads ** 3
        spatial_sum = acts.sum(dim=(1, 2), keepdim=True)
        alpha = g2 / (2 * g2 + spatial_sum * g3 + 1e-7)
        weights = (alpha * F.relu(grads)).sum(dim=(1, 2), keepdim=True)
        cam = (weights * acts).sum(dim=0).cpu().detach().numpy()
        cam = np.maximum(cam, 0)
        cam = cv2.resize(cam, (Config.IMG_SIZE, Config.IMG_SIZE))
        return (cam - cam.min()) / (cam.max() - cam.min() + 1e-7), target_class, F.softmax(output, dim=1)[0, target_class].item()

# Run Grad-CAM++ on sample test image
sample_img, sample_meta, sample_label = test_dataset[0]
img_tensor = sample_img.unsqueeze(0).to(device)
meta_tensor = sample_meta.unsqueeze(0).to(device)

gradcam = GradCAMPlusPlus(model, model.effnet.conv_head)
cam_map, pred_class, pred_conf = gradcam.generate(img_tensor, meta_tensor)

# Visualization
raw_display = sample_img.permute(1, 2, 0).cpu().numpy()
raw_display = (raw_display * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])).clip(0, 1)
heatmap = cv2.applyColorMap(np.uint8(255 * cam_map), cv2.COLORMAP_JET)
heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0
overlay = (0.6 * raw_display + 0.4 * heatmap).clip(0, 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(raw_display)
axes[0].set_title(f"Original Input (Ground Truth: {Config.CLASS_NAMES[sample_label]})")
axes[0].axis("off")

axes[1].imshow(cam_map, cmap="jet")
axes[1].set_title("Grad-CAM++ Attention Heatmap")
axes[1].axis("off")

axes[2].imshow(overlay)
axes[2].set_title(f"Clinical Overlay | Pred: {Config.CLASS_NAMES[pred_class]} ({pred_conf*100:.1f}%)")
axes[2].axis("off")

plt.tight_layout()
plt.show()
print(f"✓ Visual explanation generated: Predicted {Config.CLASS_FULL[Config.CLASS_NAMES[pred_class]]} with {pred_conf*100:.1f}% confidence.")
